# Combine per-sample AnnData objects

In this notebook, we:

- load eight preprocessed per-sample AnnData objects (`*_qc_norm.h5ad`),
- annotate each cell with `batch`, `sex`, and `condition` based on the file name, and
- concatenate all samples into a single AnnData object for downstream analysis.

No batch correction is performed in this notebook; it focuses only on concatenation and basic sanity checks.

## 1. Imports and input files

In [ ]:
import os
import numpy as np
import scanpy as sc

sc.settings.verbosity = 0
sc.settings.set_figure_params(dpi=80, facecolor="white", frameon=False)

# List of per-sample preprocessed AnnData files 
file_paths = [
    "96_6_M_KO_qc_norm.h5ad",
    "100_3_F_WT_qc_norm.h5ad",
    "114_7_F_KO_qc_norm.h5ad",
    "124_2_M_WT_qc_norm.h5ad",
    "124_4_M_KO_qc_norm.h5ad",
    "124_5_F_WT_qc_norm.h5ad",
    "148_3_M_WT_qc_norm.h5ad",
    "148_6_F_KO_qc_norm.h5ad",
]

## 2. Parse sample metadata from filenames

File names encode sample-specific information:

- `sample` (e.g. `96_6_M_KO`),
- `sex` (`M` / `F`),
- `condition` (`WT` / `KO`).

We extract these fields and store them in `.obs` for each AnnData object.

In [ ]:
def parse_sample_info(filename: str) -> dict:
    """Extract sample ID, sex, and condition from a filename like '96_6_M_KO_qc_norm.h5ad'."""
    base = os.path.basename(filename)
    parts = base.split("_")
    return {
        "sample": base.replace("_qc_norm.h5ad", ""),
        "sex": parts[2],
        "condition": parts[3].replace(".h5ad", ""),
    }

adatas = []
for file in file_paths:
    info = parse_sample_info(file)
    adata = sc.read(file)
    adata.obs["batch"] = info["sample"]
    adata.obs["sex"] = info["sex"]
    adata.obs["condition"] = info["condition"]
    adatas.append(adata)
    print(
        f"Loaded {file}: {adata.n_obs} cells, {adata.n_vars} genes, "
        f"sample={info['sample']}, sex={info['sex']}, condition={info['condition']}"
    )

## 3. Concatenate samples into a single AnnData

We concatenate all eight AnnData objects along the cell axis, keeping all genes present in any sample (`join='outer'`). Each cell already carries a `batch` label corresponding to its original sample.

In [ ]:
adata_combined = sc.concat(
    adatas,
    axis=0,
    label="batch",
    keys=[parse_sample_info(f)["sample"] for f in file_paths],
    index_unique="-",
    join="outer",  # keep all genes from all samples
)

print(
    f"Combined AnnData object has {adata_combined.n_obs} cells and "
    f"{adata_combined.n_vars} genes."
)

## 4. Gene set overlap sanity check

Before batch correction, we compare:

- the sum of per-sample gene counts,
- the number of unique genes across all samples,
- the number of genes in the combined AnnData.

This helps confirm that the concatenation behaved as expected.

In [ ]:
all_genes = set()
total_sum = 0
for adata in adatas:
    total_sum += adata.n_vars
    all_genes.update(adata.var_names)

print(f"Sum of genes in individual files: {total_sum}")
print(f"Unique genes across all samples: {len(all_genes)}")
print(f"Genes in combined AnnData: {adata_combined.n_vars}")

## 5. Save combined AnnData

We save the concatenated AnnData object for subsequent batch correction, feature selection, and dimensionality reduction steps.

In [ ]:
output_file = "combined_8_samples_all_layers_outer.h5ad"
adata_combined.write(output_file)
print(f"Combined AnnData file written: {output_file}")